In [9]:
import pandas as pd

url = "https://www.spc.noaa.gov/wcm/data/1950-2024_actual_tornadoes.csv"

df = pd.read_csv(url)

# Select 2000–2024
df = df[df["yr"].between(2000, 2024)].copy()

# Find tornadoes on the Joplin disaster date
joplin = df[
    (df["date"] == "2011-05-22")
    & (df["st"] == "MO")
]

print(joplin[
    ["date", "st", "mag", "fat", "inj", "loss"]
])

             date  st  mag  fat   inj      loss
55583  2011-05-22  MO    0    0     0     0.001
55589  2011-05-22  MO    0    0     0     0.000
55591  2011-05-22  MO    0    0     0     0.003
55598  2011-05-22  MO    1    0     0     0.230
55600  2011-05-22  MO    5  158  1150  2800.100
55604  2011-05-22  MO    2    0     0     1.000
55615  2011-05-22  MO    2    0     0     1.050


In [10]:
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from io import BytesIO

BASE_URL = (
    "https://www.ncei.noaa.gov/pub/data/"
    "swdi/stormevents/csvfiles/"
)

response = requests.get(BASE_URL, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

files = [
    a.get("href")
    for a in soup.find_all("a", href=True)
    if a.get("href", "").startswith("StormEvents_details")
    and a.get("href", "").endswith(".csv.gz")
]

all_tornadoes = []

for year in range(2000, 2027):

    matching = [
        f for f in files
        if f"_d{year}_" in f
    ]

    if not matching:
        print(f"No file found for {year}")
        continue

    # Select the latest revision
    filename = sorted(matching)[-1]

    print(f"Downloading {year}...")

    r = requests.get(BASE_URL + filename, timeout=180)
    r.raise_for_status()

    yearly_df = pd.read_csv(
        BytesIO(r.content),
        compression="gzip",
        low_memory=False
    )

    yearly_df = yearly_df[
        yearly_df["EVENT_TYPE"].eq("Tornado")
    ].copy()

    all_tornadoes.append(yearly_df)

df = pd.concat(all_tornadoes, ignore_index=True)

print("Total tornado records:", len(df))

Total tornado records: 38776


In [12]:
import pandas as pd
import numpy as np

def parse_damage(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper().replace(",", "")

    if value in ["", "NAN", "NONE", "NULL"]:
        return np.nan

    multipliers = {
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000
    }

    suffix = value[-1]

    if suffix in multipliers:
        number = value[:-1]

        if not number:
            return np.nan

        return pd.to_numeric(
            number, errors="coerce"
        ) * multipliers[suffix]

    return pd.to_numeric(value, errors="coerce")


# Convert damage values
for col in ["DAMAGE_PROPERTY", "DAMAGE_CROPS"]:

    if col in df.columns:
        new_col = col + "_NUM"

        df[new_col] = df[col].apply(parse_damage)

# Convert fatalities
for col in ["DEATHS_DIRECT", "DEATHS_INDIRECT"]:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    ).fillna(0)

# Calculate totals
df["total_deaths"] = (
    df["DEATHS_DIRECT"] + df["DEATHS_INDIRECT"]
)

df["total_damage_usd"] = df[
    ["DAMAGE_PROPERTY_NUM", "DAMAGE_CROPS_NUM"]
].sum(axis=1, min_count=1)

In [13]:
major_tornadoes = df[
    df["TOR_F_SCALE"].isin(["F4", "F5", "EF4", "EF5"])
    | (df["total_damage_usd"] >= 100_000_000)
    | (df["total_deaths"] >= 10)
].copy()

major_tornadoes = major_tornadoes.sort_values(
    "total_damage_usd",
    ascending=False
)

print("Major tornado records:", len(major_tornadoes))

Major tornado records: 241


In [14]:
columns = [
    "EVENT_ID",
    "EPISODE_ID",
    "BEGIN_DATE_TIME",
    "END_DATE_TIME",
    "STATE",
    "CZ_NAME",
    "BEGIN_LOCATION",
    "TOR_F_SCALE",
    "TOR_LENGTH",
    "TOR_WIDTH",
    "total_damage_usd",
    "total_deaths",
    "INJURIES_DIRECT",
    "BEGIN_LAT",
    "BEGIN_LON",
    "EVENT_NARRATIVE"
]

major_tornadoes = major_tornadoes[columns].copy()

major_tornadoes = major_tornadoes.rename(columns={
    "EVENT_ID": "tornado_id",
    "EPISODE_ID": "episode_id",
    "BEGIN_DATE_TIME": "start_date",
    "END_DATE_TIME": "end_date",
    "STATE": "state",
    "CZ_NAME": "county",
    "BEGIN_LOCATION": "location",
    "TOR_F_SCALE": "tornado_magnitude",
    "TOR_LENGTH": "path_length_miles",
    "TOR_WIDTH": "path_width_yards",
    "INJURIES_DIRECT": "injuries",
    "BEGIN_LAT": "start_lat",
    "BEGIN_LON": "start_lon",
    "EVENT_NARRATIVE": "event_description"
})

major_tornadoes.to_csv(
    "major_tornadoes_2000_2026.csv",
    index=False
)